# div-prune: similarity-accuracy frontier batch (2026-10-26)Purpose: produce the one number the IJPRAI rejection asked for — a comparisonin which the feedback controller is *better*, not merely harmless.The manuscript claims the controller is "no-harm", and a fixed review commentanswered that this is not a superiority claim. The gap is a missingmeasurement: inter-head similarity at the **final epoch** was logged only forthe controller, so no policy could be compared against it on equal footing.This batch logs `div_last` for every policy (baseline, fixed lambda, linear,Kendall, GradNorm, controller) and then asks:> among policies that reach a comparable similarity level, which one pays the> least accuracy?Run the cells in order. Each experiment cell is resumable: finished(dataset, arm, seed) triples are skipped, so a dropped session can be continuedby re-running the cell.

In [ ]:
# GPU check (expect a T4 or better)
!nvidia-smi

In [ ]:
# Install dependencies and pull the current code (contains the div_last logging)
!pip install -q torch-geometric hydra-core omegaconf tqdm scipy pandas

import os
if not os.path.isdir('/content/div-prune'):
    !git clone -q https://github.com/HuKejie/div-prune /content/div-prune
os.chdir('/content/div-prune')
!git log --oneline -1
!pip install -q -e . --no-deps
!ls

In [ ]:
# Smoke test: 2 seeds, 5 epochs, one fixed arm — verify div_last is populated
!python -m run.pipeline.train experiment=gat_cora experiment.regularizer=attention \
    experiment.lambda_fixed=5 experiment.runs=2 model.epochs=5 seed=0 output_dir=outputs_smoke

import pandas as pd
smoke = pd.read_csv('outputs_smoke/tables/results.csv')
print(smoke[['dataset', 'seed', 'regularizer', 'lambda_fixed', 'div_last', 'test_acc']])
assert smoke['div_last'].notna().all(), "div_last missing — code is not up to date"
print("OK: div_last recorded")

In [ ]:
# === experiment arms and resumable runner ===
# baseline        : no regularizer (reference point: high similarity, high accuracy)
# fixed{L}        : constant regularization weight, swept densely around the
#                   controller's 0.7 budget so a similarity-matched arm exists
# controller      : the contribution (integral feedback, d*=0.7, eta=0.05)
# linear/kendall/gradnorm : adaptive baselines from the revision
import os
import subprocess
from datetime import datetime

import pandas as pd

CSV_PATH = "outputs_frontier/tables/results.csv"
OUT_DIR = "outputs_frontier"

ARMS = [
    # (label, regularizer, lambda_schedule, lambda_fixed)
    ("baseline",   "none",      "fixed",      0.1),
    ("fixed0.5",   "attention", "fixed",      0.5),
    ("fixed1",     "attention", "fixed",      1.0),
    ("fixed2",     "attention", "fixed",      2.0),
    ("fixed3",     "attention", "fixed",      3.0),
    ("fixed5",     "attention", "fixed",      5.0),
    ("fixed10",    "attention", "fixed",      10.0),
    ("fixed20",    "attention", "fixed",      20.0),
    ("controller", "attention", "controller", 0.0),
    ("linear",     "attention", "linear",     0.0),
    ("kendall",    "attention", "kendall",    0.0),
    ("gradnorm",   "attention", "gradnorm",   0.0),
]

SEEDS = list(range(10))  # matches the paper protocol (10 seeds)


def run(cmd):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=False).returncode


def plog(msg):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} {msg}"
    with open("progress_log.md", "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(f"[log] {line}", flush=True)


def have_run(ds, arm):
    # True if every seed of (dataset, arm) is already in the results CSV.
    if not os.path.exists(CSV_PATH):
        return False
    df = pd.read_csv(CSV_PATH)
    reg, sched, lam = arm[1], arm[2], arm[3]
    m = (
        (df.dataset == ds)
        & (df.regularizer == reg)
        & (df.lambda_schedule == sched)
        & (df.lambda_fixed == lam)
        & (df.prune_criterion.isna())
    )
    return len(df[m].seed.unique()) >= len(SEEDS)


def run_batch(datasets, arms=ARMS):
    plog(f"START batch datasets={datasets}")
    for ds in datasets:
        for arm in arms:
            label, reg, sched, lam = arm
            if have_run(ds, arm):
                print(f"[skip] {ds} {label} already complete", flush=True)
                continue
            remaining = len(SEEDS) - _n_done(ds, arm)
            rc = run(
                f"python -m run.pipeline.train experiment=gat_cora "
                f"experiment.dataset={ds} experiment.regularizer={reg} "
                f"experiment.lambda_schedule={sched} experiment.lambda_fixed={lam} "
                f"experiment.runs={remaining} seed=0 output_dir={OUT_DIR}"
            )
            plog(f"done rc={rc} {ds} {label} (runs={remaining})")
    plog(f"END batch datasets={datasets}")


def _n_done(ds, arm):
    if not os.path.exists(CSV_PATH):
        return 0
    df = pd.read_csv(CSV_PATH)
    reg, sched, lam = arm[1], arm[2], arm[3]
    m = (
        (df.dataset == ds)
        & (df.regularizer == reg)
        & (df.lambda_schedule == sched)
        & (df.lambda_fixed == lam)
        & (df.prune_criterion.isna())
    )
    return len(df[m].seed.unique())

In [ ]:
# === STAGE 1: Cora, all arms, 10 seeds (~120 runs) ===
# Run this first and inspect the analysis cell before committing to the other
# two datasets: it already shows whether a superiority claim is available.
run_batch(["cora"])

In [ ]:
# === analysis: frontier table + similarity-matched comparison ===
# The key row is "matched": the fixed-lambda arm whose final-epoch similarity
# is closest to the controller's, compared on accuracy with a paired Wilcoxon
# test over identical seeds. If the controller wins there, the paper has a
# superiority claim that the rejection asked for.
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

df = pd.read_csv(CSV_PATH)
df = df[df.prune_criterion.isna()].copy()


def arm_of(r):
    if r.regularizer == "none":
        return "baseline"
    if r.lambda_schedule == "fixed":
        return f"fixed{r.lambda_fixed:g}"
    return r.lambda_schedule


df["arm"] = df.apply(arm_of, axis=1)


def frontier(ds):
    d = df[df.dataset == ds]
    g = (
        d.groupby("arm")
        .agg(
            n=("test_acc", "size"),
            acc=("test_acc", "mean"),
            acc_sd=("test_acc", "std"),
            div_last=("div_last", "mean"),
            div_last_sd=("div_last", "std"),
            div_best=("div_final", "mean"),
            lam_last=("lambda_last", "mean"),
        )
        .reset_index()
    )
    g["acc_pp"] = (g["acc"] * 100).round(2)
    g["acc_sd_pp"] = (g["acc_sd"] * 100).round(2)
    g["div_last"] = g["div_last"].round(3)
    return g[
        ["arm", "n", "acc_pp", "acc_sd_pp", "div_last", "div_last_sd", "div_best", "lam_last"]
    ].sort_values("div_last")


def matched_compare(ds):
    d = df[df.dataset == ds]
    ctrl = d[d.arm == "controller"]
    if ctrl.empty:
        return None
    ctrl_div = ctrl.div_last.mean()
    fixed = d[d.arm.str.startswith("fixed") & (d.arm != "baseline")]
    if fixed.empty:
        return None
    prof = fixed.groupby("arm").div_last.mean()
    best_arm = (prof - ctrl_div).abs().idxmin()
    m = fixed[fixed.arm == best_arm]
    a = ctrl.set_index("seed").test_acc
    b = m.set_index("seed").test_acc
    common = a.index.intersection(b.index)
    if len(common) < 5:
        return None
    diff = (a[common].values - b[common].values) * 100
    try:
        w = wilcoxon(a[common].values, b[common].values)
        p = w.pvalue
    except Exception:
        p = float("nan")
    return {
        "matched_arm": best_arm,
        "ctrl_div_last": round(ctrl_div, 3),
        "matched_div_last": round(m.div_last.mean(), 3),
        "delta_pp": round(diff.mean(), 2),
        "delta_ci": [round(diff.mean() - 1.96 * diff.std(ddof=1) / np.sqrt(len(diff)), 2),
                     round(diff.mean() + 1.96 * diff.std(ddof=1) / np.sqrt(len(diff)), 2)],
        "wilcoxon_p": round(p, 4) if p == p else None,
        "n_pairs": len(common),
    }


for ds in sorted(df.dataset.unique()):
    print(f"\n===== {ds.upper()} =====")
    print(frontier(ds).to_string(index=False))
    mc = matched_compare(ds)
    if mc:
        print("\nsimilarity-matched comparison (controller vs closest fixed arm):")
        for k, v in mc.items():
            print(f"  {k}: {v}")

In [ ]:
# === frontier plot: accuracy vs late-phase similarity ===
# Points to the upper-left are better (low similarity at high accuracy).
import matplotlib.pyplot as plt

for ds in sorted(df.dataset.unique()):
    g = frontier(ds)
    fig, ax = plt.subplots(figsize=(6, 4))
    for _, r in g.iterrows():
        is_ctrl = r.arm == "controller"
        ax.errorbar(
            r.div_last, r.acc_pp, xerr=r.div_last_sd, yerr=r.acc_sd_pp,
            fmt="o", color="crimson" if is_ctrl else "steelblue",
            markersize=8 if is_ctrl else 5,
        )
        ax.annotate(r.arm, (r.div_last, r.acc_pp), fontsize=8, xytext=(4, 4),
                    textcoords="offset points")
    ax.set_xlabel("late-phase inter-head similarity (final epoch)")
    ax.set_ylabel("test accuracy (%)")
    ax.set_title(f"{ds}: similarity-accuracy frontier")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# === STAGE 2: Citeseer + Pubmed (only after stage 1 looks usable) ===
run_batch(["citeseer", "pubmed"])

In [ ]:
# === export: copy the results CSV to Drive so it can be pulled back locally ===
from google.colab import drive

drive.mount('/content/drive')
!cp outputs_frontier/tables/results.csv "/content/drive/MyDrive/div-prune-frontier-results.csv"
!ls -la "/content/drive/MyDrive/div-prune-frontier-results.csv"
print("Download this file and drop it into the repo as outputs_frontier/tables/results.csv")